In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

train.head()




#  train dosyası hakkında boyut ve tür bilgisi

In [ ]:
train.shape
train.info()


# verinin incelenmesi

In [ ]:
sns.countplot(x="Survived", data=train)
plt.show()

train["Survived"].value_counts(normalize=True)


In [ ]:
sns.barplot(x="Sex", y="Survived", data=train)
plt.show()

sns.barplot(x="Pclass", y="Survived", data=train)
plt.show()
##aile büyüklüğünü hesapladık 
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] +1
sns.barplot(x="FamilySize", y="Survived", data=train)
plt.show()
##boolen ifadeci inte çeciriyoruz
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

test["HasCabin"] = test["Cabin"].notnull().astype(int)
train["HasCabin"] = train["Cabin"].notnull().astype(int)
train = train.drop("Cabin", axis=1)


# sütun silme ve veriyi bolme işlemleri 

In [ ]:
from sklearn.model_selection import train_test_split

X = train.drop("Survived", axis=1)
y = train["Survived"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# eksik verileri kapatma 

In [ ]:
age_median_test = test["Age"].median()
age_median = X_train["Age"].median()
test["Age"] = test["Age"].fillna(age_median_test)
X_train["Age"] = X_train["Age"].fillna(age_median)
X_valid["Age"] = X_valid["Age"].fillna(age_median)


X_train["HasCabin"] = train.loc[X_train.index, "HasCabin"]
X_valid["HasCabin"] = train.loc[X_valid.index, "HasCabin"]



# yaş ve cinsiyet  bilgisini ayıklıyoruz 


In [ ]:
test["AgeGroup"] = pd.cut(test["Age"], bins=[0,12,18,40,60,80], labels=[0,1,2,3,4])
X_train["AgeGroup"] = pd.cut(X_train["Age"], bins=[0,12,18,40,60,80], labels=[0,1,2,3,4])
X_valid["AgeGroup"] = pd.cut(X_valid["Age"], bins=[0,12,18,40,60,80], labels=[0,1,2,3,4])
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})
X_train["Sex"] = X_train["Sex"].map({"male": 0, "female": 1})
X_valid["Sex"] = X_valid["Sex"].map({"male": 0, "female": 1})


# kullnılmayan kolonları kaldırdım 

In [ ]:
elencekler=["Embarked","Name", "Ticket", "PassengerId"]
test = test.drop(["Embarked","Name", "Ticket", "Cabin", "PassengerId"], axis=1)
X_train = X_train.drop(elencekler, axis=1)
X_valid = X_valid.drop(elencekler, axis=1)

# model eğitimi 

In [ ]:
from sklearn.ensemble import RandomForestClassifier


model = RandomForestClassifier(
    n_estimators=100,   
    random_state=42     
)
model.fit(X_train, y_train)
test_features = test[X_train.columns] 
test_pred = model.predict(test_features)


# test tahmini ve doğruluk hesabı 

In [ ]:

# X_valid üzerinde tahmin yap
y_pred = model.predict(X_valid)
from sklearn.metrics import accuracy_score

# Doğruluk oranını hesapla
accuracy = accuracy_score(y_valid, y_pred)
print("Doğruluk:", accuracy)



# csv dosyası oluşturma 

In [ ]:


submission = pd.DataFrame({
    "PassengerId": pd.read_csv("/kaggle/input/titanic/test.csv")["PassengerId"],
    "Survived": test_pred
})

submission.to_csv("submission.csv", index=False)
